In [1]:
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import harmonypy as hm
import matplotlib as mpl
import matplotlib.pyplot as plt 
from pathlib import Path
import anndata as ad

# Optional Harmony
HAVE_HARMONY = True

# Plot defaults
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=1200, facecolor='white', fontsize=8)
mpl.rcParams['savefig.dpi'] = 1200
mpl.rcParams['savefig.bbox'] = 'tight'

# --------------------
# Parameters
# --------------------
ROOT = "/group/canc2/anson/working/cf-pbmc-bal"

CONDITION_KEY = "Condition2"
BATCH_KEY = "sampleID"
CLUSTER_KEY = "ann_lv2"

COUNTS_LAYER = "counts"

N_HVG = 2000
N_PCS = 20
N_NEIGHBORS = 15

OUTDIR = os.path.join(ROOT,"data","plots","module_scoring")
os.makedirs(OUTDIR, exist_ok=True)
sc.settings.figdir = OUTDIR

def fraction_non_integer_like(X, tol=1e-6, sample_n=200000, seed=0):
    if sp.issparse(X):
        vals = X.data
    else:
        vals = np.asarray(X).ravel()
    if vals.size == 0:
        return 0.0
    rng = np.random.default_rng(seed)
    if vals.size > sample_n:
        vals = vals[rng.choice(vals.size, size=sample_n, replace=False)]
    return float(np.mean(np.abs(vals - np.round(vals)) > tol))


def ensure_counts_layer_from_X(adata: sc.AnnData, layer_key: str = COUNTS_LAYER, tol=1e-6, max_frac_nonint=1e-4):
    if layer_key in adata.layers:
        return
    frac = fraction_non_integer_like(adata.X, tol=tol)
    print(f"fraction non-integer-like in adata.X: {frac:.6g}")
    if frac > max_frac_nonint:
        raise ValueError(
            f"adata.X is not integer-like enough to treat as raw counts (frac_nonint={frac}).\n"
            "Re-convert/export with counts preserved is recommended."
        )
    if sp.issparse(adata.X):
        Xc = adata.X.copy()
        Xc.data = np.rint(Xc.data).astype(np.int32)
        adata.layers[layer_key] = Xc
    else:
        adata.layers[layer_key] = np.rint(np.asarray(adata.X)).astype(np.int32)


def set_counts_as_X(adata: sc.AnnData, layer_key: str = COUNTS_LAYER):
    if layer_key not in adata.layers:
        raise KeyError(f"Missing layer '{layer_key}'")
    adata.X = adata.layers[layer_key].copy()
    return adata


In [2]:
ROOT = "/group/canc2/anson/working/cf-pbmc-bal"
PBMC_H5AD = os.path.join(ROOT, "data", "SCEs", "analysis", "pbmc.h5ad")
BAL_H5AD  = os.path.join(ROOT, "data", "SCEs", "analysis", "bal.h5ad")

COUNTS_LAYER = "counts"

adata_pbmc = sc.read_h5ad(PBMC_H5AD)
adata_bal = sc.read_h5ad(BAL_H5AD)

for name, ad in [("pbmc", adata_pbmc), ("bal", adata_bal)]:
    for k in [CONDITION_KEY, BATCH_KEY, CLUSTER_KEY]:
        if k not in ad.obs.columns:
            raise KeyError(f"[{name}] Expected obs column '{k}' not found.")
    print(f"{name}: shape={ad.shape} layers={list(ad.layers.keys())}")
    print(f"{name}: X min/max={float(ad.X.min())} {float(ad.X.max())}")

# Keep counts layer + set as X for concatenation
print("Ensuring counts layers...")
ensure_counts_layer_from_X(adata_pbmc, layer_key=COUNTS_LAYER)
ensure_counts_layer_from_X(adata_bal, layer_key=COUNTS_LAYER)
print("pbmc layers:", list(adata_pbmc.layers.keys()))
print("bal layers:", list(adata_bal.layers.keys()))

set_counts_as_X(adata_pbmc, COUNTS_LAYER)
set_counts_as_X(adata_bal, COUNTS_LAYER)

# Concatenate (mapping keys already define dataset labels; do NOT pass keys=...)
adata = sc.concat(
    {"PBMC": adata_pbmc, "BAL": adata_bal},
    join="outer",
    label="dataset",
    merge="same"
)

# Make obs names unique (avoids the warning you saw)
adata.obs_names_make_unique()

# Ensure counts layer exists post-concat
if COUNTS_LAYER not in adata.layers:
    # If concat dropped layers, reconstruct counts from X (should still be counts)
    ensure_counts_layer_from_X(adata, layer_key=COUNTS_LAYER)

# Set ann_lv2 as categorical with an automatically derived, stable ordering:
# PBMC_* first, then BAL_*; otherwise alphabetical.
levels = sorted(adata.obs[CLUSTER_KEY].astype(str).unique().tolist())
levels = sorted(levels, key=lambda s: (0 if s.startswith("PBMC_") else 1, s))
adata.obs[CLUSTER_KEY] = pd.Categorical(adata.obs[CLUSTER_KEY].astype(str), categories=levels, ordered=True)

sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

pbmc: shape=(110809, 31166) layers=[]
pbmc: X min/max=0.0 17935.0
bal: shape=(107598, 31717) layers=[]
bal: X min/max=0.0 6411.0
Ensuring counts layers...
fraction non-integer-like in adata.X: 0
fraction non-integer-like in adata.X: 0
pbmc layers: ['counts']
bal layers: ['counts']


/group/canc2/anson/miniconda3/envs/python310/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


normalizing counts per cell
    finished (0:00:01)


In [3]:
adata

AnnData object with n_obs × n_vars = 218407 × 32566
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Barcode', 'Capture', 'HTODemux.ID', 'HTODemux_result.orig.ident', 'HTODemux_result.nCount_RNA', 'HTODemux_result.nFeature_RNA', 'HTODemux_result.nCount_HTO', 'HTODemux_result.nFeature_HTO', 'HTODemux_result.HTO_maxID', 'HTODemux_result.HTO_secondID', 'HTODemux_result.HTO_margin', 'HTODemux_result.HTO_classification', 'HTODemux_result.HTO_classification.global', 'HTODemux_result.hash.ID', 'sampleID.HTO', 'colours.hto_colours', 'colours.sample_colours', 'colours.capture_colours', 'colours.genetic_donor_colours', 'colours.sampleID.HTO_colours', 'colours.sampleID.genetics_colours', 'vireo.donor_id', 'vireo.prob_max', 'vireo.prob_doublet', 'vireo.n_vars', 'vireo.best_singlet', 'vireo.best_doublet', 'vireo.doublet_logLikRatio', 'genetic_donor', 'sampleID.genetics', 'sampleID', 'Age', 'Sex', 'Condition', 'Bronchiectasis', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'su

In [ ]:

# -----------------------
# 1) Subset PBMC to CD8T.naive
# -----------------------
ct_key_lv2 = "ann_lv2"

adata_cdc = adata_lymph_sub[
    adata_lymph_sub.obs[ct_key_lv2].astype(str).isin(["CD4T.mem"])
].copy()

# drop unused categories (Seurat fct_drop equivalent)
for k in ["ann_lv1", "ann_lv2", "Condition2"]:
    if k in adata_cdc.obs.columns and hasattr(adata_cdc.obs[k], "cat"):
        adata_cdc.obs[k] = adata_cdc.obs[k].cat.remove_unused_categories()

# -----------------------
# 2) Gene list
# -----------------------
selected = th17_genes

selected_present = [g for g in selected if g in adata_cdc.var_names]
if len(selected_present) < len(selected):
    print("Missing genes:", set(selected) - set(selected_present))

# -----------------------
# 3) DotPlot grouped by Condition2
# -----------------------
dp = sc.pl.dotplot(
    adata_cdc,
    var_names=selected_present,
    groupby="Condition2",
    standard_scale="var",
    mean_only_expressed=True,
    dot_max=0.6,
    show=False,   # capture the object / returned dict
)

# Get the figure robustly whether dp is a dict or has .fig
if isinstance(dp, dict):
    fig = dp.get("fig", plt.gcf())
elif hasattr(dp, "fig"):
    fig = dp.fig
else:
    fig = plt.gcf()

# Find the main plot axes (heuristic: an axes without a colorbar and with some ticks/labels)
main_ax = None
for ax in fig.axes:
    if getattr(ax, "colorbar", None) is None:
        # prefer axes that have x or y ticklabels (to avoid legend-only axes)
        if (len(ax.get_xticklabels()) > 0) or (len(ax.get_yticklabels()) > 0):
            main_ax = ax
            break
# fallback
if main_ax is None:
    main_ax = fig.axes[0]

# Set the title on the main axis so it sits closer to the plot; adjust `pad` to taste
main_ax.set_title("PBMC CD4T.mem", fontsize=10, pad=24)  # try pad=6,8,10 ... smaller -> closer
plt.setp(main_ax.get_xticklabels(), fontstyle="italic")

# Keep only min/max labels on any colorbars present
for ax in fig.axes:
    cb = getattr(ax, "colorbar", None)
    if cb is not None:
        vmin, vmax = cb.vmin, cb.vmax
        cb.set_ticks([vmin, vmax])
        cb.set_ticklabels([f"{vmin:.2f}", f"{vmax:.2f}"])

# Tighten layout (you can tune rect/top if needed)
fig.tight_layout()
fig.show()

# Save figure
outdir = Path("/group/canc2/anson/working/cf-pbmc-bal/data/plots/PBMC_CD4T.mem")
outdir.mkdir(parents=True, exist_ok=True)
out = os.path.join(outdir, f"PBMC-CD4T.mem_th17dotplot.jpeg")
fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")


### Prepare gene sets and datasets

In [4]:
IFN_genes = ["ABCE1","ADAR","BST2","CACTIN","CDC37","CNOT7","DCST1","EGR1","FADD","GBP2","HLA-A","HLA-B","HLA-C","HLA-E","HLA-F","HLA-G","HLA-H","HSP90AB1","IFI27","IFI35","IFI6","IFIT1","IFIT2","IFIT3","IFITM1","IFITM2","IFITM3","IFNA1","IFNA10","IFNA13","IFNA14","IFNA16","IFNA17","IFNA2","IFNA21","IFNA4","IFNA5","IFNA6","IFNA7","IFNA8","IFNAR1","IFNAR2","IFNB1","IKBKE","IP6K2","IRAK1","IRF1","IRF2","IRF3","IRF4","IRF5","IRF6","IRF7","IRF8","IRF9","ISG15","ISG20","JAK1","LSM14A","MAVS","METTL3","MIR21","MMP12","MUL1","MX1","MX2","MYD88","NLRC5","OAS1","OAS2","OAS3","OASL","PSMB8","PTPN1","PTPN11","PTPN2","PTPN6","RNASEL","RSAD2","SAMHD1","SETD2","SHFL","SHMT2","SP100","STAT1","STAT2","TBK1","TREX1","TRIM56","TRIM6","TTLL12","TYK2","UBE2K","USP18","WNT5A","XAF1","YTHDF2","YTHDF3","ZBP1"]

IFNG_genes = ["NR1H3","ACTR3","ACTR2","CDC42EP2","SYNCRIP","ADAMTS13","CDC37","VPS26B","GBP4","GBP5","OTOP1","SIRPA","DAPK1","DAPK3","GBP6","EDN1","AIF1","PDE12","EPRS1","FCAR","FLNB","RPL13A","CDC42EP4","STXBP4","GAPDH","GBP1","GBP2","GBP3","GSN","HCK","HLA-DPA1","HPX","RAB7B","RAB43","IRF8","IRGM","IFNG","IFNGR1","IFNGR2","FASLG","IL12B","IL12RB1","AQP4","IRF1","JAK1","JAK2","KIF5B","ARG1","GBP7","LGALS9","CIITA","CD200","MRC1","ASS1","MYO1C","NOS2","PIM1","PARP14","PPARG","MED1","RAB20","PTPN2","RAF1","RPS6KB1","CCL2","CCL3","CCL5","SLC26A6","SP100","STAT1","STX4","STXBP1","STXBP2","STXBP3","CRIPTO","TLR2","TLR3","TLR4","ACTG1","TNF","TP53","TXK","TYK2","ACOD1","NR1H2","VIM","WAS","WNT5A","ZYX","CALM1","CAMK2A","CASP1","PARP9","NLRC5","VAMP4","CLDN1","DNAJA3","VAMP3","STX8","CD47","CD58","CDC42"]

#TGFB_genes = ["CDH3","CDH5","TGFBR3L","SPRY1","SPRY2","STUB1","CDKN1C","CDKN2B","CITED2","TAB1","LEFTY1","GIPC1","FERMT2","EMILIN1","STRAP","IL17F","CIDEA","VASN","LRG1","WFIKKN1","WFIKKN2","COL1A2","COL3A1","CD109","CREB1","CREBBP","PARP1","DAB2","SPRED1","EID2","DAND5","SPRED2","FLCN","ENG","HTRA4","EP300","MECOM","FBN1","FBN2","FKBP1A","SNW1","PEG10","FMOD","FNTA","SIRT1","FOLR1","FOS","LEMD3","FAM89B","FSHB","FUT8","NPNT","BAMBI","BRMS1","SIN3A","TSKU","ZNF451","APPL1","LRRC32","GCNT2","LATS2","MSTN","GDF9","GDF10","AMHR2","DKK3","GLG1","GOT1","BCL9L","RGCC","HIPK2","ERO1A","HDAC1","HDAC2","ONECUT1","HPGD","HSPA1A","HSPA5","HSP90AB1","APOA1","ID1","ING1","ING2","ITGA3","ITGB1","ITGB5","ITGB6","ITGB8","JUN","NRROS","SPRED3","LOX","LTBP1","LTBP2","LTBP3","MIRLET7A1","MIRLET7B","MIRLET7F1","MIRLET7G","MIR101-1","MIR106A","MIR140","MIR142","MIR145","MIR15B","MIR17","MIR18A","MIR181A2","MIR183","MIR199A1","MIR19A","MIR19B1","MIR20A","MIR204","MIR21","MIR212","MIR26A1","MIR27B","MIR29B1","MIR30A","MIR30B","MIR9-1","MIR93","MIR98","SMAD1","SMAD2","SMAD3","SMAD4","ARRB2","SMAD5","SMAD6","SMAD7","SMAD9","MEN1","MIR302B","MIR323A","MIR342","MIR376C","MIR372","MIR373","CITED1","NDP","NODAL","MIR361","MIR424","FURIN","CHST11","ZBTB7A","TRIM33","PDPK1","NLK","ARID4B","PIN1","PML","PPARA","PPARG","IL17RD","RNF111","ASPN","PPM1A","ADISSP","APPL2","FERMT1","INTS9","HTRA1","PSG9","PMEPA1","DUSP22","TWSG1","SMURF1","ZMIZ1","SELENON","MIR490","MIR497","MIR498","MIR520C","PTK2","PTPRK","PXN","OVOL2","SINHCAF","SNX6","ARID4A","RBBP4","RBBP7","BCL9","SDCBP","PRDM16","PBLD","FKBP1C","PALS1","SUDS3","SMURF2","SKI","SKIL","BMP2","SKOR2","BMPR1A","RASL11B","SOX11","SPI1","SRC","STAT3","STK11","ADAM17","MAP3K7","MIR564","ZEB1","TGFB1","TGFB1I1","TGFB2","TGFB3","TGFBR1","TGFBR2","TGFBR3","THBS1","NKX2-1","CLDN5","TP53","WNT1","XBP1","LDLRAD4","ZYX","SAP130","VEPH1","SAP30L","ZNF703","TET1","SLC2A10","GDF5","USP9X","USP9Y","AXIN1","ZMIZ2","SNX25","LTBP4","BRMS1L","OGT","CILP","ITGA8","CAV2","CAV3","ADAM9","SAP30","TSC22D1","FOXH1","ACVR1","PIAS2","MTMR4","LATS1","NREP","MYOCD","ZFYVE9","TGFBRAP1","ACVRL1","HTRA3","LPXN","ONECUT2","GDF15","ADAMTSL2","ZEB2","USP15"]
TGFB_genes = ["CDH3","CDH5","TGFBR3L","DNM3OS","SPRY1","SPRY2","STUB1","CDKN1C","CDKN2B","CITED2","TAB1","POSTN","LEFTY1","GIPC1","FERMT2","EMILIN1","STRAP","IL17F","CIDEA","VASN","LRG1","WFIKKN1","WFIKKN2","COL1A1","COL1A2","COL3A1","COL4A2","CD109","CREB1","CREBBP","CRK","CRKL","PARP1","DAB2","SPRED1","EID2","DLX1","EDN1","DAND5","SPRED2","FLCN","ENG","HTRA4","EP300","MECOM","FBN1","FBN2","FGFR2","FKBP1A","SNW1","PEG10","FMOD","FNTA","SIRT1","FOLR1","FOS","LEMD3","CD2AP","FAM89B","FSHB","ABL1","FUT8","FYN","NPNT","BAMBI","BRMS1","MXRA5","SIN3A","TSKU","ZNF451","APPL1","LRRC32","GCNT2","LATS2","MSTN","GDF9","GDF10","AMHR2","ANKRD1","DKK3","GLG1","GOT1","BCL9L","RGCC","HIPK2","NR3C1","ERO1A","HDAC1","HDAC2","APAF1","ONECUT1","HPGD","HSPA1A","HSPA5","HSP90AB1","APOA1","ID1","IL4","ING1","ING2","ISL1","ITGA3","ITGB1","ITGB5","ITGB6","ITGB8","JUN","NRROS","LIMS1","SPRED3","LOX","LTBP1","LTBP2","LTBP3","MIRLET7A1","MIRLET7B","MIRLET7F1","MIRLET7G","MIR101-1","MIR106A","MIR130A","MIR140","MIR142","MIR145","MIR15B","MIR17","MIR18A","MIR181A2","MIR183","MIR199A1","MIR19A","MIR19B1","MIR20A","MIR204","MIR21","MIR212","MIR26A1","MIR27A","MIR27B","MIR29B1","MIR30A","MIR30B","MIR9-1","MIR93","MIR98","SMAD1","SMAD2","SMAD3","SMAD4","ARRB2","SMAD5","SMAD6","SMAD7","SMAD9","MEF2C","MEN1","KMT2A","MIR302B","MIR323A","MIR342","MIR376C","MIR372","MIR373","CITED1","ZFHX3","NDP","NODAL","DDR2","MIR361","MIR424","FURIN","CHST11","PAK2","ZBTB7A","PDE2A","PDE3A","TRIM33","PDPK1","NLK","WWOX","ARID4B","PENK","PIN1","PML","WNT4","PPARA","PPARG","IL17RD","RNF111","ASPN","PPM1A","ADISSP","APPL2","SOX6","FERMT1","INTS9","MAPK7","HTRA1","PSG9","PMEPA1","DUSP22","TWSG1","SMURF1","ZMIZ1","SELENON","MIR490","MIR497","MIR498","MIR520C","PTK2","EPB41L5","PTPRK","PXN","OVOL2","SINHCAF","SNX6","ACTA2","ARID4A","RBBP4","RBBP7","BCL9","ROCK1","XCL1","SDCBP","PRDM16","PBLD","SFRP1","FKBP1C","SCX","NFKBIZ","PALS1","SUDS3","SMURF2","FNDC4","SKI","SKIL","BMP2","SKOR2","BMPR1A","RASL11B","SNRNP70","SOX5","SOX9","SOX11","SPI1","SRC","ZFP36L1","STAT3","ZFP36L2","STK11","ADAM17","MAP3K7","MIR564","ZEB1","TGFB1","TGFB1I1","TGFB2","TGFB3","TGFBR1","TGFBR2","TGFBR3","THBS1","NKX2-1","CLDN5","TP53","WNT1","WNT2","WNT5A","WNT7A","XBP1","YES1","LDLRAD4","ZYX","SAP130","VEPH1","SAP30L","ZNF703","PDGFD","TET1","WNT10A","SLC2A10","GDF5","USP9X","USP9Y","AXIN1","ZMIZ2","SNX25","LTBP4","BRMS1L","OGT","CILP","ITGA8","CAV1","STK16","CAV2","CAV3","RUNX3","HYAL2","ADAM9","SAP30","CFLAR","TSC22D1","FOXH1","ACVR1","PIAS2","CLDN1","MTMR4","LATS1","PDCD5","NREP","MYOCD","ZFYVE9","TGFBRAP1","ACVRL1","HTRA3","LPXN","ROCK2","ONECUT2","GDF15","ADAMTSL2","ZEB2","USP15"]
#TGFB_genes = ["ACVR1","APC","ARID4B","BCAR3","BMP2","BMPR1A","BMPR2","CDH1","CDK9","CDKN1C","CTNNB1","ENG","FKBP1A","FNTA","FURIN","HDAC1","HIPK2","ID1","ID2","ID3","IFNGR2","JUNB","KLF10","LEFTY2","LTBP2","MAP3K7","NCOR2","NOG","PMEPA1","PPM1A","PPP1CA","PPP1R15A","RAB31","RHOA","SERPINE1","SKI","SKIL","SLC20A1","SMAD1","SMAD3","SMAD6","SMAD7","SMURF1","SMURF2","SPTBN1","TGFB1","TGFBR1","TGIF1","THBS1","TJP1","TRIM33","UBE2D3","WWTR1","XIAP"]

th17_genes = ["BATF","TRAF3IP2","ARID5A","MALT1","SLAMF6","RC3H1","IL23R","EP300","EPHB2","BRD4","IL17RA","TBX21","NLRP10","IL2","IL6","IL6R","IL6ST","IL12B","IL12RB1","IRF4","JAK1","JAK2","JAK3","JUNB","LGALS1","LY9","MIR21","SMAD7","ASCL2","NOTCH1","OPA1","FOXP3","ZBTB7B","IL23A","PHB1","RC3H2","PRKCQ","ENTPD7","BRD2","RORA","RORC","CARD9","NFKBIZ","CLEC7A","STAT3","STAT5A","TYK2","ZC3H12A","LOXL3","NFKBID","TNFSF18","SOCS3","CLEC6A","IL27RA","CD69"]

trm_genes = ["CD69","ITGAE","CXCR6","ITGA1","CD101","ZNF683","PRDM1","ENTPD1","LGALS1","VIM","ANXA1","LMNA", "RGS1"]


In [5]:
# =============================================================================
# Module scoring with scanpy's sc.tl.score_genes (single-cell)
# -> score CELLS per module (scanpy control-gene matching)
# -> pseudobulk by DONOR: mean(cell scores) within ann_lv2 × Condition2 × sampleID
# -> stats: MWU/Wilcoxon rank-sum on DONOR means (CFB vs C, CFN vs C) per ann_lv2
# -> display: 3 heatmaps (IFN/JAK/TNF)
#    - color = mean donor score (optionally z-scored per module for display)
#    - stars = significance (donor-level MWU vs C)
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu
from statsmodels. stats.multitest import multipletests
import scanpy as sc

# -------------------------
# Settings
# -------------------------
CLUSTER_KEY   = "ann_lv2"
CONDITION_KEY = "Condition2"
DONOR_KEY     = "sampleID"

cond_colors = {"C": "#26547c", "CFN":  "#ffd166", "CFB": "#ef476f"}

baseline    = "C"
cond_order  = ["C", "CFB", "CFN"]
comparisons = ["CFB", "CFN", "CFB_vs_CFN"]

exclude_lv2 = {}

levels=[
    "PBMC_CD4T.naive",
    "PBMC_CD8T.naive",
    "PBMC_CD4T.mem",
    "PBMC_CD8T.mem",
    "PBMC_Treg",
    "PBMC_MAIT",
    "PBMC_gdT",
    "PBMC_NKT",
    "PBMC_NK",
    "PBMC_ILC",
    "PBMC_B",
    "BAL_CD4T.naive",
    "BAL_CD4T.mem",
    "BAL_CD8T.mem",
    "BAL_Treg",
    "BAL_ILC",
    "BAL_NKT",
    "BAL_NK",
    "BAL_B"
]
celltypes_preferred = levels

# -------------------------
# Existing modules
# -------------------------
modules = ["IFN", "IFNG", "th17", "TRM", "TGFB"]

module_to_gene_list = {
    "IFN": IFN_genes,
    "IFNG": IFNG_genes,
    "th17":  th17_genes,
    "TRM": trm_genes,
    "TGFB": TGFB_genes
}

module_title = {
    "IFN": "Type I IFN response",
    "IFNG": "Type II IFN response",
    "th17": "Th17 response",
    "TRM":  "Tissue residency",
    "TGFB": r"TGF$\beta$ response"
}

# =============================================================================
# Load and append Szabo2025 T cell gene sets
# =============================================================================

print("Loading Szabo2025 gene sets...")

# Load the gene set table
geneset_path = "/group/canc2/anson/working/cf-pbmc-bal/data/Szabo2025_Tcell_geneSet.csv"
gs_df = pd.read_csv(geneset_path)

# Get available genes from your adata objects
genes_in_obj = set(adata.var_names)

# Process each column (module) from the CSV
szabo_modules = []
for col in gs_df.columns:
    # Extract genes from column, remove NAs and empty strings
    genes = gs_df[col].dropna().astype(str).str.strip()
    genes = genes[genes != '']. unique().tolist()
    
    # Keep only genes present in the object
    genes = [g for g in genes if g in genes_in_obj]
    
    # Take top 50 genes
    #genes = genes[:50]
    
    # Only keep modules with at least some genes
    if len(genes) > 0:
        # Create a module key (you can modify naming convention as needed)
        module_key = col.replace(" ", "_").replace("/", "_")
        
        # Append to existing dictionaries
        module_to_gene_list[module_key] = genes
        module_title[module_key] = col  # Use original column name as title
        szabo_modules.append(module_key)
        
        print(f"  Added {module_key}:  {len(genes)} genes")

# Update modules list to include Szabo modules
modules.extend(szabo_modules)

print(f"\nTotal modules:  {len(modules)}")
print(f"  Original:  {['IFN', 'IFNG', 'th17', 'TRM']}")
print(f"  Szabo2025: {szabo_modules[: 5]}...  ({len(szabo_modules)} total)")

# =============================================================================
# Optional: Select specific Szabo modules if you don't want all 18
# =============================================================================
# # If you only want specific modules from Szabo, uncomment and modify: 
szabo_modules_to_include = [
    "Inflammatory_Cytokine",
    #"IFN_Response", 
    "Cytotoxicity",
    "Naive_CM_Homing",
    "Tissue_Signature"
]

# Filter to only include selected modules
modules = ["IFN", "IFNG", "th17"] + szabo_modules_to_include

# # Remove unwanted modules from dictionaries
# all_keys = list(module_to_gene_list.keys())
# for key in all_keys:
#     if key not in modules: 
#         del module_to_gene_list[key]
#         del module_title[key]

modules

Loading Szabo2025 gene sets...
  Added Inflammatory_Cytokine:  97 genes
  Added Activ._Survival:  98 genes
  Added Iron_Metabolism:  99 genes
  Added IFN_Response:  99 genes
  Added Treg:  98 genes
  Added Naive_CM_Homing:  93 genes
  Added Resting_Inhibition:  96 genes
  Added Cytotoxicity:  98 genes
  Added Activ._Metabol.:  98 genes
  Added Glycolysis:  97 genes
  Added Naive_CM_Quiescence:  95 genes
  Added Tissue_Signature:  100 genes
  Added Gut_Residency:  95 genes
  Added Immunoregulatory:  98 genes
  Added Chemokine_Cytotoxic:  97 genes
  Added Resting_T_cell:  96 genes
  Added CD8_T_cell_Rest:  94 genes
  Added Infant_Tissue:  95 genes

Total modules:  23
  Original:  ['IFN', 'IFNG', 'th17', 'TRM']
  Szabo2025: ['Inflammatory_Cytokine', 'Activ._Survival', 'Iron_Metabolism', 'IFN_Response', 'Treg']...  (18 total)


['IFN',
 'IFNG',
 'th17',
 'Inflammatory_Cytokine',
 'Cytotoxicity',
 'Naive_CM_Homing',
 'Tissue_Signature']

### Scoring

In [6]:
SCOREGENES_CTRL_SIZE = 50
SCOREGENES_N_BINS = 25
SCOREGENES_USE_RAW = False

# -------------------------
# Helpers
# -------------------------
def stars_from_q(q):
    if pd.isna(q): return ""
    if q < 1e-3: return "***"
    if q < 1e-2: return "**"
    if q < 5e-2: return "*"
    return ""

# -------------------------
# Calculate Module Scores for Each Dataset
# -------------------------
print(f"\nProcessing...")
for mod in modules:
    genes = [g for g in module_to_gene_list[mod] if g in adata.var_names]
    if len(genes) < 5:
        print(f"{mod}: too few genes present for scoring (present={len(genes)}). Skipping.")
        continue
    sc.tl.score_genes(
        adata,
        gene_list=genes,
        score_name=f"{mod}_score",
        ctrl_size=len(genes), #SCOREGENES_CTRL_SIZE,
        n_bins=SCOREGENES_N_BINS,
        use_raw=SCOREGENES_USE_RAW,
        random_state=0,
    )



Processing...
computing score 'IFN_score'
    finished: added
    'IFN_score', score of gene set (adata.obs).
    1934 total control genes are used. (0:00:05)
computing score 'IFNG_score'
    finished: added
    'IFNG_score', score of gene set (adata.obs).
    1891 total control genes are used. (0:00:05)
computing score 'th17_score'
    finished: added
    'th17_score', score of gene set (adata.obs).
    740 total control genes are used. (0:00:05)
computing score 'Inflammatory_Cytokine_score'
    finished: added
    'Inflammatory_Cytokine_score', score of gene set (adata.obs).
    1454 total control genes are used. (0:00:05)
computing score 'Cytotoxicity_score'
    finished: added
    'Cytotoxicity_score', score of gene set (adata.obs).
    1163 total control genes are used. (0:00:05)
computing score 'Naive_CM_Homing_score'
    finished: added
    'Naive_CM_Homing_score', score of gene set (adata.obs).
    1110 total control genes are used. (0:00:05)
computing score 'Tissue_Signature_

### Rename

In [7]:
# Single-adata lymphoid subset with dataset-aware prefixed labels
# Requires:
# - adata.obs["dataset"] with values like {"PBMC","BAL"}
# - CLUSTER_KEY points to your original ann_lv2 / clustering labels
# Excludes: anything containing "Prolif"

KEEP_KEYS = ("CD4T", "CD8T", "Treg", "gdT", "MAIT", "ILC", "NK", "NKT","B")
DROP_KEY  = "Prolif"

def is_naive_label(x: str) -> bool:
    return "naive" in str(x).lower()

def is_trm_label(x: str) -> bool:
    return "trm" in str(x).lower()

def rename_lymphoid(dataset: str, x: str) -> str | None:
    ds = str(dataset)
    x = str(x)

    if DROP_KEY.lower() in x.lower():
        return None

    # PBMC rules
    if ds == "PBMC":
        if x.startswith("B."):
            return "B"
        if "CD4T" in x:
            return "CD4T.naive" if is_naive_label(x) else "CD4T.mem"
        if "CD8T" in x:
            return "CD8T.naive" if is_naive_label(x) else "CD8T.mem"

        if "Treg" in x: return "Treg"
        if "gdT" in x:  return "gdT"
        if "MAIT" in x: return "MAIT"
        if "NKT" in x:  return "NKT"
        if "ILC" in x:  return "ILC"
        if "NK.CD56hi" in x:  return "NK"
        if "NK.CD56lo" in x:  return "NK.CD56lo"
        return None

    # BAL rules
    if ds == "BAL":
        if x.startswith("B"):
            return "B"
        if "CD4T" in x:
            if is_naive_label(x): return "CD4T.naive"
            if is_trm_label(x):   return "CD4T.TRM"
            return "CD4T.mem"
        if "CD8T" in x:
            if is_naive_label(x): return "CD8T.naive"
            if is_trm_label(x):   return "CD8T.TRM"
            return "CD8T.mem"

        if "Treg" in x: return "Treg"
        if "gdT" in x:  return "gdT"
        if "MAIT" in x: return "MAIT"
        if "NKT" in x:  return "NKT"
        if "ILC" in x:  return "ILC"
        if x == "NK" or x.startswith("NK"):
            return "NK"
        return None

    # unknown dataset -> drop
    return None

# --- build keep mask on the combined adata ---
lv2 = adata.obs[CLUSTER_KEY].astype(str)
ds  = adata.obs["dataset"].astype(str)

keep_mask = (
    lv2.str.contains("|".join(KEEP_KEYS), regex=True, na=False)
    | ((ds == "PBMC") & lv2.str.startswith("NK.", na=False))  # keep PBMC NK.* specifically
)
keep_mask &= ~lv2.str.contains(DROP_KEY, case=False, na=False)

adata_lymph_sub = adata[keep_mask].copy()

# --- apply dataset-aware renaming into the same column (or choose a new one) ---
adata_lymph_sub.obs[CLUSTER_KEY] = [
    rename_lymphoid(d, x)
    for d, x in zip(adata_lymph_sub.obs["dataset"].astype(str),
                    adata_lymph_sub.obs[CLUSTER_KEY].astype(str))
]

# drop anything that didn't map
adata_lymph_sub = adata_lymph_sub[adata_lymph_sub.obs[CLUSTER_KEY].notna()].copy()

print("Subset counts by dataset and new label:")
print(adata_lymph_sub.obs.groupby(["dataset", CLUSTER_KEY]).size().sort_values(ascending=False))

Subset counts by dataset and new label:
dataset  ann_lv2   
PBMC     CD4T.naive    38901
         B             16747
         CD8T.naive    13106
         NK.CD56lo     10951
         CD4T.mem       8069
         CD8T.mem       4653
         gdT            4649
BAL      CD8T.TRM       2892
PBMC     Treg           2762
BAL      B              2442
         CD4T.mem       2234
PBMC     NKT            1674
         NK             1513
BAL      CD4T.naive     1510
PBMC     MAIT           1406
BAL      NK              973
         CD8T.mem        707
         NKT             678
         Treg            647
         CD4T.TRM        463
         ILC             287
PBMC     ILC             142
         CD8T.TRM          0
         CD4T.TRM          0
BAL      MAIT              0
         CD8T.naive        0
         NK.CD56lo         0
         gdT               0
dtype: int64


/tmp/ipykernel_2729611/1509440440.py:89: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(adata_lymph_sub.obs.groupby(["dataset", CLUSTER_KEY]).size().sort_values(ascending=False))


### Pseudobulk scores

In [8]:
# -------------------------
# Pseudobulk: Mean Scores by Donor
# -------------------------
donor_scores = (
    adata_lymph_sub.obs
    .assign(
        ann_lv2=lambda d: d['dataset'].astype(str) + "_" + d[CLUSTER_KEY].astype(str),
        Condition2=lambda d: d[CONDITION_KEY].astype(str),
        sampleID=lambda d: d[DONOR_KEY].astype(str),
    )
    .groupby(["ann_lv2", "Condition2", "sampleID"], dropna=False)
    [[f"{m}_score" for m in modules]]
    .mean()
    .reset_index()
)
donor_scores

,ann_lv2,Condition2,sampleID,IFN_score,IFNG_score,th17_score,Inflammatory_Cytokine_score,Cytotoxicity_score,Naive_CM_Homing_score,Tissue_Signature_score
0,BAL_B,C,M1N066,0.188634,0.195759,0.048132,0.053346,0.002145,-0.035639,0.168235
1,BAL_B,C,M1N075,0.227034,0.241443,0.068793,0.094837,0.036931,-0.035159,0.183305
2,BAL_B,C,M1N078,0.208638,0.190421,0.051312,0.076935,0.001642,-0.039397,0.096695
3,BAL_B,C,M1N080,0.191302,0.197962,0.107644,0.114897,-0.000509,-0.036861,0.050943
4,BAL_B,C,M1N087,0.162849,0.189152,0.056473,0.063992,0.014618,-0.026155,0.189443
...,...,...,...,...,...,...,...,...,...,...
373,PBMC_gdT,CFN,M1C180D,0.241478,0.201724,0.185725,0.134822,0.365567,-0.014556,0.209912
374,PBMC_gdT,CFN,M1C191B,0.216456,0.176260,0.110182,0.100269,0.359547,-0.021794,0.167407
375,PBMC_gdT,CFN,M1C199B,0.271997,0.228337,0.178169,0.128299,0.395272,-0.009278,0.241494
376,PBMC_gdT,CFN,M1C201A,0.263427,0.230721,0.156388,0.113336,0.431120,0.013147,0.213381


In [10]:
import pandas as pd
from pathlib import Path

# Keep only these cell types
keep_cell_types = ["PBMC_Mono", "BAL_Mono"]

# Filter first
df_filt = donor_scores[donor_scores["ann_lv2"].isin(keep_cell_types)].copy()

# Long format
id_cols = ["ann_lv2", "Condition2", "sampleID"]
long_df = (
    df_filt.melt(
        id_vars=id_cols,
        var_name="Pathway",
        value_name="Score",
    )
    .rename(columns={"ann_lv2": "Cell type"})
)

# Optional: remove trailing "_score"
long_df["Pathway"] = long_df["Pathway"].str.replace(r"_score$", "", regex=True)

# Exact column order
long_df = long_df[["Cell type", "Condition2", "sampleID", "Pathway", "Score"]]

# Export
out_dir = Path("/group/canc2/anson/working/cf-pbmc-bal/data")
out_path = out_dir / "myeloid_donor_scores.xlsx"
out_dir.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    long_df.to_excel(writer, sheet_name="long_scores", index=False)

print(f"Saved: {out_path}")

Saved: /group/canc2/anson/working/cf-pbmc-bal/data/lymphoid_donor_scores.xlsx


In [9]:
# unique ann_lv2 levels (sorted)
levels = (
    donor_scores["ann_lv2"]
    .dropna()
    .astype(str)
    .sort_values()
    .unique()
)

print(f"Number of ann_lv2 levels: {len(levels)}")
for x in levels:
    print(x)

Number of ann_lv2 levels: 22
BAL_B
BAL_CD4T.TRM
BAL_CD4T.mem
BAL_CD4T.naive
BAL_CD8T.TRM
BAL_CD8T.mem
BAL_ILC
BAL_NK
BAL_NKT
BAL_Treg
PBMC_B
PBMC_CD4T.mem
PBMC_CD4T.naive
PBMC_CD8T.mem
PBMC_CD8T.naive
PBMC_ILC
PBMC_MAIT
PBMC_NK
PBMC_NK.CD56lo
PBMC_NKT
PBMC_Treg
PBMC_gdT


In [10]:
# -------------------------
# Statistical Testing: MWU/Wilcoxon rank-sum (donor means)
# Single concatenated donor_scores (no dataset split):
#   loop: ann_lv2 (celltype) × module × pairwise comparison
#   BH correction within each (ann_lv2 × module) panel across the 3 pairwise p-values
# -------------------------

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

PAIRWISE = [
    ("CFB", "C",   "CFB vs C",   "CFB"),
    ("CFN", "C",   "CFN vs C",   "CFN"),
    ("CFB", "CFN", "CFB vs CFN", "CFB_vs_CFN"),
]

# modules should be like: ["TNF","IFNG","MHC","JAK","TGFB","cytokine","granulocyte_migration","mono"]
# and columns in donor_scores must be e.g. TNF_score, IFNG_score, ...

required_cols = {"ann_lv2", "Condition2", "sampleID"}
missing = required_cols - set(donor_scores.columns)
if missing:
    raise ValueError(f"donor_scores is missing required columns: {sorted(missing)}")

# ensure types
d = donor_scores.copy()
d["ann_lv2"] = d["ann_lv2"].astype(str)
d["Condition2"] = d["Condition2"].astype(str)
d["sampleID"] = d["sampleID"].astype(str)

stat_rows = []

for ct in d["ann_lv2"].dropna().unique():
    d_ct = d[d["ann_lv2"] == ct]

    for mod in modules:
        col = f"{mod}_score"
        if col not in d_ct.columns:
            continue

        for g1, g2, label, key in PAIRWISE:
            x = d_ct.loc[d_ct["Condition2"] == g1, col].astype(float).dropna().values
            y = d_ct.loc[d_ct["Condition2"] == g2, col].astype(float).dropna().values

            if (len(x) < 2) or (len(y) < 2):
                p = np.nan
            else:
                p = mannwhitneyu(x, y, alternative="two-sided").pvalue

            stat_rows.append({
                "ann_lv2": ct,
                "module": mod,
                "comparison": label,   # human label
                "Condition2": key,     # key used for star_map / plotting
                "group1": g1,
                "group2": g2,
                "p": p,
                "n_group1_donors": int(len(x)),
                "n_group2_donors": int(len(y)),
            })

stats_grp = pd.DataFrame(stat_rows)
stats_grp["q"] = np.nan

# -------------------------
# BH adjust within each panel (ann_lv2 × module) across the 3 pairwise p-values
# -------------------------
for (ct, mod), sub_idx in stats_grp.groupby(["ann_lv2", "module"]).groups.items():
    sub_idx = list(sub_idx)
    pvals = stats_grp.loc[sub_idx, "p"].values
    m = ~np.isnan(pvals)
    if m.sum() == 0:
        continue

    _, qvals, *_ = multipletests(pvals[m], method="fdr_bh")
    q_full = np.full_like(pvals, np.nan, dtype=float)
    q_full[m] = qvals
    stats_grp.loc[sub_idx, "q"] = q_full

def stars_from_q(q):
    if pd.isna(q): return ""
    if q < 1e-3: return "***"
    if q < 1e-2: return "**"
    if q < 5e-2: return "*"
    return ""

stats_grp["stars"] = stats_grp["q"].map(stars_from_q)

# star_map includes all 3 comparisons
star_map = {
    (r.ann_lv2, r.module, r.Condition2): (r.stars if isinstance(r.stars, str) else "")
    for r in stats_grp.itertuples(index=False)
}

# -------------------------
# Print significant results
# -------------------------
sig = stats_grp[stats_grp["q"].notna() & (stats_grp["q"] < 0.05)].copy()
if len(sig):
    print(
        sig.sort_values(["comparison", "q", "ann_lv2", "module"])[
            ["ann_lv2","module","comparison","p","q","stars","n_group1_donors","n_group2_donors"]
        ].to_string(index=False)
    )
else:
    print("No significant results at q < 0.05 (panel-wise BH).")

     ann_lv2                module comparison        p        q stars  n_group1_donors  n_group2_donors
   PBMC_MAIT                   IFN   CFB vs C 0.004329 0.012987     *                6                5
   PBMC_MAIT                  IFNG   CFB vs C 0.004329 0.012987     *                6                5
BAL_CD8T.TRM                   IFN CFB vs CFN 0.002165 0.006494    **                6                6
BAL_CD8T.TRM                  IFNG CFB vs CFN 0.004329 0.012987     *                6                6
     BAL_ILC          Cytotoxicity CFB vs CFN 0.007937 0.012987     *                5                5
    PBMC_ILC          Cytotoxicity CFB vs CFN 0.004329 0.012987     *                6                5
BAL_CD8T.mem Inflammatory_Cytokine CFB vs CFN 0.017316 0.025974     *                5                6
BAL_CD8T.TRM                   IFN   CFN vs C 0.008658 0.012987     *                6                6
     BAL_ILC          Cytotoxicity   CFN vs C 0.008658 0.012987 

### Box plot of mean module scores

In [13]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Assumes these already exist:
# donor_scores (ann_lv2, Condition2, sampleID, <mod>_score...)
# stats_grp (dataset, ann_lv2, module, Condition2, p.adj/p_adj or q)
# cond_order, baseline, module_title, ROOT

cond_color = {"C": "#26547c", "CFN": "#ffd166", "CFB": "#ef476f"}

# -------------------------
# Choose what to plot (lists)
# -------------------------
CT_PLOT_LIST  = [#"PBMC_CD4T.naive","PBMC_CD8T.naive","PBMC_CD4T.mem","PBMC_CD8T.mem",
                 "PBMC_MAIT","PBMC_ILC",
                 #"PBMC_Treg","PBMC_gdT","PBMC_MAIT","BAL_CD4T.mem","BAL_CD8T.TRM","BAL_NK","BAL_ILC"
]   # <-- edit
MOD_PLOT_LIST = ["IFN","IFNG","Cytotoxicity"]                 # <-- edit (module keys)
ALL_COMBINATIONS = True  # True = every ct × every mod; False = zip(ct, mod)


# -------------------------
# Helpers
# -------------------------
def stars_from_padj(p):
    if pd.isna(p): return ""
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 5e-2: return "*"
    return ""

def get_ds_from_ct(ct: str) -> str:
    if str(ct).startswith("PBMC_"):
        return "PBMC"
    if str(ct).startswith("BAL_"):
        return "BAL"
    return "PBMC"

def format_ct_title(ct: str) -> str:
    ds = get_ds_from_ct(ct)
    ct_clean = str(ct).replace("PBMC_", "").replace("BAL_", "")
    return f"{ds} {ct_clean}"

def sanitize_filename(s: str) -> str:
    # keep letters/numbers/._- ; replace others with _
    out = []
    for ch in str(s):
        if ch.isalnum() or ch in "._-":
            out.append(ch)
        else:
            out.append("_")
    return "".join(out)

def _get_padj(stats_grp, ds, ct, mod, cond_key):
    rr = stats_grp[
        (stats_grp.get("dataset", ds) == ds) &
        (stats_grp["ann_lv2"].astype(str) == str(ct)) &
        (stats_grp["module"].astype(str) == str(mod)) &
        (stats_grp["Condition2"].astype(str) == str(cond_key))
    ]
    if len(rr) == 0:
        return np.nan
    if "p.adj" in rr.columns:
        return float(rr["p.adj"].iloc[0])
    if "p_adj" in rr.columns:
        return float(rr["p_adj"].iloc[0])
    if "q" in rr.columns:
        return float(rr["q"].iloc[0])
    return np.nan

def _add_sig_bracket(ax, x1, x2, y, text, barh, lw=1.2, text_pad=0.02):
    """
    text_pad is a fraction of barh (0.25 means place text 0.25*barh above the bracket)
    """
    ax.plot([x1, x1, x2, x2], [y, y + barh, y + barh, y], c="black", lw=lw, clip_on=False)
    ax.text(
        (x1 + x2) / 2,
        y + barh * (text_pad-1),   # <- move text above the bracket line
        text,
        ha="center",
        va="bottom",
        fontsize=12
    )

def _box_with_aligned_points(ax, df, order, colors):
    """
    Match your attached style:
    - box unfilled (transparent), colored outline
    - no jitter; points vertically aligned
    - median line colored to match group
    """
    data = [df.loc[df["Condition2"] == c, "score"].astype(float).values for c in order]

    bp = ax.boxplot(
        data,
        labels=order,
        patch_artist=True,
        showfliers=False,
        widths=0.55,
        medianprops=dict(linewidth=1.6),   # color set below per group
        boxprops=dict(linewidth=1.6),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
    )

    # boxes + whiskers/caps + medians colored by group
    for i, c in enumerate(order):
        col = colors.get(c, "black")

        # box
        bp["boxes"][i].set_facecolor("none")
        bp["boxes"][i].set_edgecolor(col)
        bp["boxes"][i].set_linewidth(1.6)

        # median line for this box
        bp["medians"][i].set_color(col)
        bp["medians"][i].set_linewidth(1.8)

        # whiskers/caps come in pairs per box
        for w in bp["whiskers"][2*i:2*i+2]:
            w.set_color(col)
            w.set_linewidth(1.2)
        for cap in bp["caps"][2*i:2*i+2]:
            cap.set_color(col)
            cap.set_linewidth(1.2)

    # aligned points (no jitter)
    for i, c in enumerate(order, start=1):
        y = df.loc[df["Condition2"] == c, "score"].astype(float).values
        x = np.full_like(y, i, dtype=float)
        ax.scatter(
            x, y, s=28,
            color=colors.get(c, "gray"),
            edgecolor="black",
            linewidth=0.4,
            zorder=3
        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def plot_box_for_ct_mod(
    *,
    donor_scores: pd.DataFrame,
    stats_grp: pd.DataFrame,
    ct: str,
    mod: str,
    cond_order,
    module_title: dict,
    outdir: str,
    colors: dict
):
    ds = get_ds_from_ct(ct)

    score_col = f"{mod}_score"
    if score_col not in donor_scores.columns:
        print(f"SKIP: {mod} | {ct} -> missing column {score_col}")
        return

    df_ct = donor_scores[donor_scores["ann_lv2"].astype(str).eq(str(ct))].copy()
    if df_ct.empty:
        print(f"SKIP: {mod} | {ct} -> no donor_scores rows")
        return

    dfm = (
        df_ct[["sampleID", "Condition2", score_col]]
        .rename(columns={score_col: "score"})
        .dropna()
    )
    if dfm.empty:
        print(f"SKIP: {mod} | {ct} -> all NaN scores")
        return

    present = set(dfm["Condition2"].astype(str))
    order = [c for c in cond_order if c in present]
    if len(order) < 2:
        print(f"SKIP: {mod} | {ct} -> <2 conditions present: {order}")
        return

    fig, ax = plt.subplots(1, 1, figsize=(3.1, 3.2))
    _box_with_aligned_points(ax, dfm, order=order, colors=colors)

    # Title: not bold; add dataset before celltype
    title = f"{module_title.get(mod, mod)}\n{format_ct_title(ct)}"
    ax.set_title(title, fontsize=13, fontweight="normal", pad=8)

    ax.set_ylabel("Mean module score")
    ax.set_xlabel("")

    # Add asterisk brackets based on p.adj only (no numeric label)
    comparisons_to_draw = [
        ("C", "CFB", "CFB"),            # CFB vs C
        ("C", "CFN", "CFN"),            # CFN vs C
        ("CFB", "CFN", "CFB_vs_CFN"),   # CFB vs CFN (if computed)
    ]

    y = dfm["score"].astype(float).values
    ymax = np.nanmax(y)
    ymin = np.nanmin(y)
    yrng = (ymax - ymin) if np.isfinite(ymax - ymin) and (ymax > ymin) else 1.0
    base_y = ymax + 0.10 * yrng
    step_y = 0.08 * yrng
    barh = 0.02 * yrng

    xpos = {cond: i + 1 for i, cond in enumerate(order)}

    n_added = 0
    for a, b, cond_key in comparisons_to_draw:
        if a not in xpos or b not in xpos:
            continue

        padj = _get_padj(stats_grp, ds, ct, mod, cond_key)
        stars = stars_from_padj(padj)
        if stars == "":
            continue

        yy = base_y + n_added * step_y
        _add_sig_bracket(ax, xpos[a], xpos[b], y=yy, text=stars, barh=barh, lw=1.2)
        n_added += 1

    ax.set_ylim(ymin - 0.05 * yrng, base_y + (n_added + 0.6) * step_y)

    plt.tight_layout()

    os.makedirs(outdir, exist_ok=True)
    # Output name: module first, then celltype
    mod_fn = sanitize_filename(mod)
    ct_fn = sanitize_filename(ct)
    out = os.path.join(outdir, f"{mod_fn}_{ct_fn}.jpeg")
    fig.savefig(out, dpi=1600, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("Saved:", out)


# -------------------------
# Loop and plot
# -------------------------
if ALL_COMBINATIONS:
    for ct in CT_PLOT_LIST:
        outdir = os.path.join(ROOT, "data", "plots", "boxplots_selected",ct)
        os.makedirs(outdir, exist_ok=True)
        for mod in MOD_PLOT_LIST:
            plot_box_for_ct_mod(
                donor_scores=donor_scores,
                stats_grp=stats_grp,
                ct=ct,
                mod=mod,
                cond_order=cond_order,
                module_title=module_title,
                outdir=outdir,
                colors=cond_color,
            )
else:
    if len(CT_PLOT_LIST) != len(MOD_PLOT_LIST):
        raise ValueError("When ALL_COMBINATIONS=False, CT_PLOT_LIST and MOD_PLOT_LIST must have same length.")
    for ct, mod in zip(CT_PLOT_LIST, MOD_PLOT_LIST):
        plot_box_for_ct_mod(
            donor_scores=donor_scores,
            stats_grp=stats_grp,
            ct=ct,
            mod=mod,
            cond_order=cond_order,
            module_title=module_title,
            outdir=outdir,
            colors=cond_color,
        )

print("Done. Output dir:", outdir)

/tmp/ipykernel_716702/2923515981.py:96: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/boxplots_selected/PBMC_MAIT/IFN_PBMC_MAIT.jpeg


/tmp/ipykernel_716702/2923515981.py:96: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/boxplots_selected/PBMC_MAIT/IFNG_PBMC_MAIT.jpeg


/tmp/ipykernel_716702/2923515981.py:96: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


...

Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/boxplots_selected/PBMC_ILC/IFN_PBMC_ILC.jpeg


/tmp/ipykernel_716702/2923515981.py:96: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/boxplots_selected/PBMC_ILC/IFNG_PBMC_ILC.jpeg


/tmp/ipykernel_716702/2923515981.py:96: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/boxplots_selected/PBMC_ILC/Cytotoxicity_PBMC_ILC.jpeg
Done. Output dir: /group/canc2/anson/working/cf-pbmc-bal/data/plots/boxplots_selected/PBMC_ILC


### Heatmap

In [31]:
# -------------------------
# Define celltypes from both subsets
# -------------------------
celltypes = ["PBMC_MAIT","PBMC_ILC"]

# -------------------------
# Heatmap Values: Mean of Donor Means (per subtype × condition)
# -------------------------
score_cols = ["IFN_score","IFNG_score"]

display_means = (
    donor_scores
    .groupby(["ann_lv2", "Condition2"], dropna=False)[score_cols]
    .mean()
    .reset_index()
)

disp_long = display_means.melt(
    id_vars=["ann_lv2", "Condition2"],
    value_vars=score_cols,
    var_name="module_score",
    value_name="score",
)
disp_long["module"] = disp_long["module_score"].str.replace("_score", "", regex=False)

# Add dataset column (match your established heatmap chunk)
disp_long["dataset"] = np.where(
    disp_long["ann_lv2"].astype(str).str.startswith("PBMC_"), "PBMC",
    np.where(disp_long["ann_lv2"].astype(str).str.startswith("BAL_"), "BAL", "PBMC")
)

# Optional: z-score per module across all subtypes/conditions (for visualization)
disp_long["score_z"] = np.nan
for mod in modules:
    mm = disp_long["module"].eq(mod)
    v = disp_long.loc[mm, "score"].astype(float).values
    mu = np.nanmean(v)
    sd = np.nanstd(v)
    disp_long.loc[mm, "score_z"] = (disp_long.loc[mm, "score"].astype(float) - mu) / (sd + 1e-8)

# -------------------------
# Plot: Heatmaps grouped by Cell Type (One Per Cell Type)
# -------------------------
n_celltypes = len(celltypes)
n_cols = 3
n_rows = int(np.ceil(n_celltypes / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, 2.5 * n_rows), sharey=True)
axes = axes.ravel()
fig.subplots_adjust(hspace=0.45, top=0.92)

cmap = plt.get_cmap("PuOr_r")
vmin, vmax = -2, 2

x_edges = np.arange(len(cond_order) + 1)
y_edges = np.arange(len(modules) + 1)
x_centers = np.arange(len(cond_order)) + 0.5
y_centers = np.arange(len(modules)) + 0.5

mappable = None

for ax_idx, ct in enumerate(celltypes):
    ax = axes[ax_idx]

    # Decide dataset exactly like your established chunk
    if str(ct).startswith("PBMC_"):
        ds = "PBMC"
    elif str(ct).startswith("BAL_"):
        ds = "BAL"
    else:
        ds = "PBMC"

    subct = disp_long[(disp_long["dataset"] == ds) & (disp_long["ann_lv2"] == ct)].copy()

    # Build matrix: rows = modules, columns = conditions
    d = {(r.module, r.Condition2): r.score_z for r in subct.itertuples(index=False)}
    Z = np.full((len(modules), len(cond_order)), np.nan, dtype=float)
    for rr, mod in enumerate(modules):
        for cc, cond in enumerate(cond_order):
            Z[rr, cc] = d.get((mod, cond), np.nan)

    pm = ax.pcolormesh(x_edges, y_edges, Z, cmap=cmap, vmin=vmin, vmax=vmax, shading="flat")
    if mappable is None:
        mappable = pm

    # Match established formatting
    ax.set_xticks(np.arange(len(cond_order) + 1), minor=True)
    ax.set_yticks(np.arange(len(modules) + 1), minor=True)
    ax.grid(which="major", color="white", linewidth=0.2)
    ax.tick_params(which="major", bottom=False, left=False)

    ax.set_title(ct, fontsize=9, fontweight="normal")
    ax.set_xlim(0, len(cond_order))
    ax.set_ylim(0, len(modules))

    ax.set_xticks(x_centers)
    ax.set_xticklabels(cond_order, rotation=0)

    ax.set_yticks(y_centers)
    ax.set_yticklabels([module_title[m] for m in modules])

    # Stars on non-baseline cells only (vs baseline)
    # Match established star_map signature: (ds, ct, mod, cond)
    for rr, mod in enumerate(modules):
        for cc, cond in enumerate(cond_order):
            if cond == baseline:
                continue
            st = star_map.get((ds, ct, mod, cond), "")
            if st:
                ax.text(cc + 0.5, rr + 0.5, st, ha="center", va="center",
                        fontsize=8, color="black")

# Hide unused subplots
for ax_idx in range(n_celltypes, len(axes)):
    axes[ax_idx].axis("off")

# Colorbar: match established settings
cbar = fig.colorbar(
    mappable,
    ax=axes.ravel().tolist(),
    fraction=0.04,
    pad=0.2,
    location="right",
    shrink=0.3,
    anchor=(4.0, 0.5),
    panchor=(4.0, 0.5),
    aspect=10
)
cbar.set_label("Module z-score")

plt.tight_layout()
plt.show()

out = os.path.join(OUTDIR, "ModuleScore_MWU.jpeg")
fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")
print("Saved:", out)

/tmp/ipykernel_696669/2974990398.py:37: RuntimeWarning: Mean of empty slice
  mu = np.nanmean(v)
/group/canc2/anson/miniconda3/envs/python310/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_696669/2974990398.py:37: RuntimeWarning: Mean of empty slice
  mu = np.nanmean(v)
/group/canc2/anson/miniconda3/envs/python310/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_696669/2974990398.py:37: RuntimeWarning: Mean of empty slice
  mu = np.nanmean(v)
/group/canc2/anson/miniconda3/envs/python310/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_696669/2974990398.p

Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/module_scoring/ModuleScore_MWU.jpeg


## Paired t test

### First round

In [ ]:
# Optimized + flexible (works with 3 CTs or 4 CTs)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests

alpha = 0.05
min_donors = 2

assert "donor_scores" in globals(), "donor_scores must exist."
assert "modules" in globals()
conds_to_plot = ["CFB", "CFN"] # "C", 
cond_color = {"C": "#26547c", "CFN": "#ffd166", "CFB": "#ef476f"}
assert "stars_from_q" in globals()

# -----------------------------
# 1) Configure CTs flexibly
# -----------------------------
# Put ANY CTs you want here (3 or 4 or more). Adjacent comparisons are tested/plotted.          # example: 3 CTs
ct_list = ["CD4T.mem","NK","ILC"]
for ct in ct_list:
    cts = [f"PBMC_{ct}",f"BAL_{ct}"]
    
    # Labels must match length of cts
    ct_label_map = {
        f"PBMC_{ct}":  f"PBMC\n{ct}",
        f"BAL_{ct}":  f"BAL\n{ct}",
    }
    x_labels = [ct_label_map.get(c, c) for c in cts]
    
    # Auto x positions (even spacing)
    x_span = 0.2   # was effectively ~0.66; smaller => CTs closer together
    x_ticks = np.linspace(0.0, x_span, num=len(cts))
    x_map = dict(zip(cts, x_ticks))
    x_pad = 0.06
    
    # Adjacent comparisons (families)
    adj_pairs = list(zip(cts[:-1], cts[1:]))
    
    # Optional per-condition rule: skip first pair for "C" (like your original)
    def pairs_for_condition(cond):
        if cond == "C":
            return adj_pairs[1:]  # skip (cts[0] -> cts[1])
        return adj_pairs
    
    # -----------------------------
    # 2) Precompute long form once
    # -----------------------------
    score_cols = [f"{m}_score" for m in modules]
    plot_long = donor_scores.melt(
        id_vars=["ann_lv2", "Condition2", "sampleID"],
        value_vars=score_cols,
        var_name="module_score",
        value_name="score",
    )
    plot_long["module"] = plot_long["module_score"].str.replace("_score", "", regex=False)
    
    # -----------------------------
    # 3) Precompute wide pivots ONCE per (module, condition)
    # -----------------------------
    # pivots[(module, cond)] = wide df: index=sampleID, columns=ann_lv2, values=score
    pivots = {}
    for mod in modules:
        dmod = plot_long[plot_long["module"].eq(mod)]
        for cond in conds_to_plot:
            d = dmod[dmod["Condition2"].eq(cond)]
            piv = d.pivot_table(
                index="sampleID", columns="ann_lv2", values="score",
                aggfunc="mean", observed=False
            )
            # ensure requested CT columns exist (in correct order), missing => NaN
            piv = piv.reindex(columns=cts)
            pivots[(mod, cond)] = piv
    
    # -----------------------------
    # 4) Compute all adjacent paired tests (BH per comparison family)
    # -----------------------------
    rows = []
    for mod in modules:
        for cond in conds_to_plot:
            piv = pivots[(mod, cond)]
            for (ctA, ctB) in adj_pairs:
                piv2 = piv.dropna(subset=[ctA, ctB], how="any")
                n = int(piv2.shape[0])
                if n < 2:
                    t, p = np.nan, np.nan
                else:
                    t, p = ttest_rel(piv2[ctA].to_numpy(), piv2[ctB].to_numpy(), nan_policy="omit")
                rows.append({
                    "Condition2": cond, "module": mod,
                    "ctA": ctA, "ctB": ctB,
                    "comparison": f"{ctA} vs {ctB}",
                    "n_donors": n, "t": t, "p": p
                })
    
    stats_all = pd.DataFrame(rows)
    
    # BH adjust *within each adjacent-pair family* (ctA,ctB), across all (cond,module)
    stats_all["p_adj"] = np.nan
    for (ctA, ctB), subidx in stats_all.groupby(["ctA", "ctB"]).groups.items():
        m = stats_all.loc[subidx, "p"].notna()
        if m.sum() > 0:
            stats_all.loc[subidx[m], "p_adj"] = multipletests(stats_all.loc[subidx[m], "p"], method="fdr_bh")[1]
    
    stats_all["stars"] = stats_all["p_adj"].map(stars_from_q)
    
    # Fast lookup: (cond, mod, ctA, ctB) -> (p_adj, stars, n)
    pmap = {
        (r.Condition2, r.module, r.ctA, r.ctB): (r.p_adj, r.stars, r.n_donors)
        for r in stats_all.itertuples(index=False)
    }
    
    # -----------------------------
    # 5) Plot
    # -----------------------------
    outdir = os.path.join(ROOT, "data", "plots")
    os.makedirs(outdir, exist_ok=True)
    
    for mod in modules:
        # -----------------------------
        # y-limits from ACTUAL plotted data (selected CTs + selected conditions),
        # plus extra headroom for significance bars/text.
        # -----------------------------
        vals_for_ylim = []
        for cond in conds_to_plot:
            piv = pivots[(mod, cond)].copy()
            if piv.shape[0] == 0:
                continue
            n_present = piv.notna().sum(axis=1)
            piv_plot = piv.loc[n_present >= 2]   # same rule as plotting
            if piv_plot.shape[0] == 0:
                continue
            vals_for_ylim.append(piv_plot.to_numpy().ravel())
        
        if len(vals_for_ylim) == 0:
            y_min, y_max = -0.1, 0.1
        else:
            vv = np.concatenate(vals_for_ylim)
            vv = vv[np.isfinite(vv)]
            if vv.size == 0:
                y_min, y_max = -0.1, 0.1
            else:
                # use strict min/max (or swap to quantiles if you prefer robustness)
                y_min, y_max = float(np.min(vv)), float(np.max(vv))
        
        data_range = (y_max - y_min) if (y_max > y_min) else 1.0
        
        # how many bracket rows could appear at most in a panel?
        # (you only draw adjacent pairs; you then filter to significant ones, but reserve worst-case space)
        max_brackets = max(len(pairs_for_condition(c)) for c in conds_to_plot) if conds_to_plot else 0
        
        # spacing must match your annotation spacing so the reserved headroom is accurate
        spacing = 0.06 * data_range
        tick_h = spacing * 0.15
        text_pad = spacing * 0.35  # extra for the star text above the line
        
        top_headroom = max_brackets * spacing + tick_h + text_pad
        bottom_pad = 0.08 * data_range
        
        y_lim = (y_min - bottom_pad, y_max + top_headroom)
    
        fig, axes = plt.subplots(1, len(conds_to_plot), figsize=(2 * len(conds_to_plot), 2.5), sharey=True)
        axes = np.atleast_1d(axes)
        fig.subplots_adjust(wspace=0.25, top=0.86)
    
        for ax, cond in zip(axes, conds_to_plot):
            piv = pivots[(mod, cond)].copy()
    
            if piv.shape[0] == 0:
                ax.text(0.5, 0.5, "no donors", transform=ax.transAxes, ha="center", va="center")
                ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, fontsize=9)
                ax.set_title(cond, fontsize=11); ax.set_ylim(*y_lim)
                continue
    
            # donors with >=2 present points among chosen CTs
            n_present = piv.notna().sum(axis=1)
            piv_plot = piv.loc[n_present >= 2].sort_index()
    
            col = cond_color.get(cond, "black")
            for donor, row in piv_plot.iterrows():
                ys_all = row.to_numpy()
                mask = ~np.isnan(ys_all)
                xs = x_ticks[mask]
                ys = ys_all[mask]
                ax.plot(xs, ys, color="0.75", linewidth=1.0, zorder=1)
                ax.scatter(xs, ys, s=22, color=col, edgecolor="black", linewidth=0.3, zorder=2)
    
            ax.set_title(cond, fontsize=11)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels(x_labels, fontsize=9)
            ax.set_xlim(np.min(x_ticks) - x_pad, np.max(x_ticks) + x_pad)
            ax.set_ylim(*y_lim)
            import matplotlib.ticker as mtick
            ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
    
            # --- Significant annotations (only for allowed pairs for this condition) ---
            y_top = y_lim[1]
            # stack only the significant ones, so gaps don't appear when one comparison isn't significant
            sig_pairs = []
            for (ctA, ctB) in pairs_for_condition(cond):
                p_adj, stars, n_d = pmap.get((cond, mod, ctA, ctB), (np.nan, "", 0))
                if (n_d is not None) and (n_d >= min_donors) and (pd.notna(p_adj)) and (p_adj < alpha):
                    sig_pairs.append((ctA, ctB, p_adj, stars, n_d))
    
            spacing = 0.06 * data_range
            for j, (ctA, ctB, p_adj, stars, n_d) in enumerate(sig_pairs):
                x1, x2 = x_map[ctA], x_map[ctB]
                y_line = y_top - (j + 1) * spacing
                tick_h = spacing * 0.15
    
                ax.plot([x1, x2], [y_line, y_line], color="k", linewidth=1.0, zorder=3)
                ax.plot([x1, x1], [y_line, y_line - tick_h], color="k", linewidth=1.0, zorder=3)
                ax.plot([x2, x2], [y_line, y_line - tick_h], color="k", linewidth=1.0, zorder=3)
    
                star_text = stars.strip() if isinstance(stars, str) else ""
                #txt = f"{star_text} p.adj={p_adj:.3f}".strip()
                txt = f"{star_text}".strip()
                ax.text((x1 + x2) / 2, y_line + spacing * 0.12, txt,
                        ha="center", va="bottom", fontsize=8, zorder=4)
    
        axes[0].set_ylabel("Mean module score")
        fig.suptitle(f"{module_title.get(mod, mod)}\n{ct}", fontsize=13, y=0.98)
    
        fig.tight_layout()
        # os.makedirs(os.path.join(OUTDIR,ct), exist_ok=True)
        out = os.path.join(OUTDIR, ct, f"pairedT_{mod}_PBMC-{ct}_vs_BAL-{ct}_forCS.jpeg")
        # fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")
        #plt.show()
        print("Saved:", out)
    
    # -----------------------------
    # 6) Optional: show significant rows
    # -----------------------------
    def show_significant(stats_df, alpha=0.05, min_donors=2):
        sig = stats_df[(stats_df["p_adj"].notna()) & (stats_df["p_adj"] < alpha) & (stats_df["n_donors"] >= min_donors)]
        if sig.empty:
            print(f"No significant results (p_adj < {alpha}).")
            return
        cols = ["Condition2", "module", "comparison", "n_donors", "t", "p", "p_adj", "stars"]
        display(sig.loc[:, cols].sort_values(["Condition2", "module", "comparison"]).reset_index(drop=True))
    
    show_significant(stats_all, alpha=alpha, min_donors=min_donors)

In [16]:
from pathlib import Path
import pandas as pd

# Choose where to save (uses your requested location)
out_dir = Path("/group/canc2/anson/working/cf-pbmc-bal/data")
out_dir.mkdir(parents=True, exist_ok=True)

# stats_all exists inside the ct loop; ct is the current cell-type label (e.g., "CD4T.mem")
# Write one file per ct (recommended), or change to a constant filename if you prefer to overwrite.
out_path = out_dir / f"paired_ttest_statistics_PBMC_vs_BAL_{ct}.xlsx"

cols = [c for c in ["Condition2", "module", "ctA", "ctB", "comparison", "n_donors", "t", "p", "p_adj", "stars"] if c in stats_all.columns]
stats_export = (
    stats_all.loc[:, cols]
    .sort_values(["Condition2", "module", "ctA", "ctB"])
    .reset_index(drop=True)
)

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    stats_export.to_excel(writer, sheet_name="paired_ttests", index=False)

print(f"Saved stats xlsx: {out_path}")

Saved stats xlsx: /group/canc2/anson/working/cf-pbmc-bal/data/paired_ttest_statistics_PBMC_vs_BAL_ILC.xlsx


In [ ]:
# Optimized + flexible (works with 3 CTs or 4 CTs)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests
from pathlib import Path

alpha = 0.05
min_donors = 2

assert "donor_scores" in globals(), "donor_scores must exist."
assert "modules" in globals()
assert "stars_from_q" in globals()
assert "module_title" in globals(), "module_title must exist (used in plot titles)."
assert "ROOT" in globals(), "ROOT must exist (used to build plot output path)."
assert "OUTDIR" in globals(), "OUTDIR must exist (used to build plot output path)."

conds_to_plot = ["CFB", "CFN"]  # "C",
cond_color = {"C": "#26547c", "CFN": "#ffd166", "CFB": "#ef476f"}

# -----------------------------
# 1) Configure CTs flexibly
# -----------------------------
ct_list = ["CD4T.mem", "NK", "ILC"]

# Precompute long form ONCE (no need to redo per ct)
score_cols = [f"{m}_score" for m in modules]
plot_long = donor_scores.melt(
    id_vars=["ann_lv2", "Condition2", "sampleID"],
    value_vars=score_cols,
    var_name="module_score",
    value_name="score",
)
plot_long["module"] = plot_long["module_score"].str.replace("_score", "", regex=False)

# Accumulate per-ct stats here, then export one xlsx at the end
all_stats = []

for ct in ct_list:
    cts = [f"PBMC_{ct}", f"BAL_{ct}"]

    ct_label_map = {
        f"PBMC_{ct}": f"PBMC\n{ct}",
        f"BAL_{ct}": f"BAL\n{ct}",
    }
    x_labels = [ct_label_map.get(c, c) for c in cts]

    # Auto x positions (even spacing)
    x_span = 0.2
    x_ticks = np.linspace(0.0, x_span, num=len(cts))
    x_map = dict(zip(cts, x_ticks))
    x_pad = 0.06

    # Adjacent comparisons (families)
    adj_pairs = list(zip(cts[:-1], cts[1:]))

    def pairs_for_condition(cond):
        if cond == "C":
            return adj_pairs[1:]  # skip first pair if you ever include "C"
        return adj_pairs

    # -----------------------------
    # 2) Precompute wide pivots ONCE per (module, condition) for this ct
    # -----------------------------
    pivots = {}
    for mod in modules:
        dmod = plot_long[plot_long["module"].eq(mod)]
        for cond in conds_to_plot:
            d = dmod[dmod["Condition2"].eq(cond)]
            piv = d.pivot_table(
                index="sampleID", columns="ann_lv2", values="score",
                aggfunc="mean", observed=False
            )
            piv = piv.reindex(columns=cts)  # ensure requested CT columns exist
            pivots[(mod, cond)] = piv

    # -----------------------------
    # 3) Compute all adjacent paired tests (BH per comparison family)
    # -----------------------------
    rows = []
    for mod in modules:
        for cond in conds_to_plot:
            piv = pivots[(mod, cond)]
            for (ctA, ctB) in adj_pairs:
                piv2 = piv.dropna(subset=[ctA, ctB], how="any")
                n = int(piv2.shape[0])
                if n < 2:
                    t, p = np.nan, np.nan
                else:
                    t, p = ttest_rel(
                        piv2[ctA].to_numpy(),
                        piv2[ctB].to_numpy(),
                        nan_policy="omit",
                    )
                rows.append({
                    "ct": ct,
                    "Condition2": cond,
                    "module": mod,
                    "ctA": ctA,
                    "ctB": ctB,
                    "comparison": f"{ctA} vs {ctB}",
                    "n_donors": n,
                    "t": t,
                    "p": p,
                })

    stats_all = pd.DataFrame(rows)

    # BH adjust *within each adjacent-pair family* (ctA,ctB), across all (cond,module) for this ct
    stats_all["p_adj"] = np.nan
    for (ctA, ctB), subidx in stats_all.groupby(["ctA", "ctB"]).groups.items():
        m = stats_all.loc[subidx, "p"].notna()
        if m.sum() > 0:
            stats_all.loc[subidx[m], "p_adj"] = multipletests(
                stats_all.loc[subidx[m], "p"],
                method="fdr_bh",
            )[1]

    stats_all["stars"] = stats_all["p_adj"].map(stars_from_q)

    # Lookup for plotting annotations
    pmap = {
        (r.Condition2, r.module, r.ctA, r.ctB): (r.p_adj, r.stars, r.n_donors)
        for r in stats_all.itertuples(index=False)
    }

    # Accumulate for final single xlsx
    all_stats.append(stats_all)

    # -----------------------------
    # 4) Plot (same as your logic)
    # -----------------------------
    outdir = os.path.join(ROOT, "data", "plots")
    os.makedirs(outdir, exist_ok=True)

    for mod in modules:
        vals_for_ylim = []
        for cond in conds_to_plot:
            piv = pivots[(mod, cond)].copy()
            if piv.shape[0] == 0:
                continue
            n_present = piv.notna().sum(axis=1)
            piv_plot = piv.loc[n_present >= 2]
            if piv_plot.shape[0] == 0:
                continue
            vals_for_ylim.append(piv_plot.to_numpy().ravel())

        if len(vals_for_ylim) == 0:
            y_min, y_max = -0.1, 0.1
        else:
            vv = np.concatenate(vals_for_ylim)
            vv = vv[np.isfinite(vv)]
            if vv.size == 0:
                y_min, y_max = -0.1, 0.1
            else:
                y_min, y_max = float(np.min(vv)), float(np.max(vv))

        data_range = (y_max - y_min) if (y_max > y_min) else 1.0
        max_brackets = max(len(pairs_for_condition(c)) for c in conds_to_plot) if conds_to_plot else 0

        spacing = 0.06 * data_range
        tick_h = spacing * 0.15
        text_pad = spacing * 0.35
        top_headroom = max_brackets * spacing + tick_h + text_pad
        bottom_pad = 0.08 * data_range
        y_lim = (y_min - bottom_pad, y_max + top_headroom)

        fig, axes = plt.subplots(
            1, len(conds_to_plot),
            figsize=(2 * len(conds_to_plot), 2.5),
            sharey=True
        )
        axes = np.atleast_1d(axes)
        fig.subplots_adjust(wspace=0.25, top=0.86)

        for ax, cond in zip(axes, conds_to_plot):
            piv = pivots[(mod, cond)].copy()

            if piv.shape[0] == 0:
                ax.text(0.5, 0.5, "no donors", transform=ax.transAxes, ha="center", va="center")
                ax.set_xticks(x_ticks)
                ax.set_xticklabels(x_labels, fontsize=9)
                ax.set_title(cond, fontsize=11)
                ax.set_ylim(*y_lim)
                continue

            n_present = piv.notna().sum(axis=1)
            piv_plot = piv.loc[n_present >= 2].sort_index()

            col = cond_color.get(cond, "black")
            for donor, row in piv_plot.iterrows():
                ys_all = row.to_numpy()
                mask = ~np.isnan(ys_all)
                xs = x_ticks[mask]
                ys = ys_all[mask]
                ax.plot(xs, ys, color="0.75", linewidth=1.0, zorder=1)
                ax.scatter(xs, ys, s=22, color=col, edgecolor="black", linewidth=0.3, zorder=2)

            ax.set_title(cond, fontsize=11)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels(x_labels, fontsize=9)
            ax.set_xlim(np.min(x_ticks) - x_pad, np.max(x_ticks) + x_pad)
            ax.set_ylim(*y_lim)

            import matplotlib.ticker as mtick
            ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.2f"))
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            # Significant annotations
            y_top = y_lim[1]
            sig_pairs = []
            for (ctA, ctB) in pairs_for_condition(cond):
                p_adj, stars, n_d = pmap.get((cond, mod, ctA, ctB), (np.nan, "", 0))
                if (n_d is not None) and (n_d >= min_donors) and (pd.notna(p_adj)) and (p_adj < alpha):
                    sig_pairs.append((ctA, ctB, p_adj, stars, n_d))

            for j, (ctA, ctB, p_adj, stars, n_d) in enumerate(sig_pairs):
                x1, x2 = x_map[ctA], x_map[ctB]
                y_line = y_top - (j + 1) * spacing

                ax.plot([x1, x2], [y_line, y_line], color="k", linewidth=1.0, zorder=3)
                ax.plot([x1, x1], [y_line, y_line - tick_h], color="k", linewidth=1.0, zorder=3)
                ax.plot([x2, x2], [y_line, y_line - tick_h], color="k", linewidth=1.0, zorder=3)

                star_text = stars.strip() if isinstance(stars, str) else ""
                ax.text(
                    (x1 + x2) / 2,
                    y_line + spacing * 0.12,
                    f"{star_text}".strip(),
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    zorder=4,
                )

        axes[0].set_ylabel("Mean module score")
        fig.suptitle(f"{module_title.get(mod, mod)}\n{ct}", fontsize=13, y=0.98)

        fig.tight_layout()
        out = os.path.join(OUTDIR, ct, f"pairedT_{mod}_PBMC-{ct}_vs_BAL-{ct}_forCS.jpeg")
        # fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")
        # plt.close(fig)
        print("Saved:", out)

# -----------------------------
# 5) Export ONE combined xlsx across all CTs
# -----------------------------
out_dir = Path("/group/canc2/anson/working/cf-pbmc-bal/data")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "paired_ttest_statistics_PBMC_vs_BAL_all_celltypes.xlsx"

if len(all_stats) == 0:
    print("No stats to export (all_stats is empty).")
else:
    stats_concat = pd.concat(all_stats, ignore_index=True)

    export_cols = [
        "ct", "Condition2", "module", "ctA", "ctB", "comparison",
        "n_donors", "t", "p", "p_adj", "stars"
    ]
    export_cols = [c for c in export_cols if c in stats_concat.columns]

    stats_export = (
        stats_concat.loc[:, export_cols]
        .sort_values(["ct", "Condition2", "module", "ctA", "ctB"])
        .reset_index(drop=True)
    )

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        stats_export.to_excel(writer, sheet_name="paired_ttests", index=False)

    print(f"Saved combined stats xlsx: {out_path}")